# Decorrelation-penalised soft-constrained PCA

`factors.soft_constrained_pca` fits the prior patterns jointly but
lets the implied score time series collapse onto the same market
direction (loading correlation up to ~99% on the company data, ~53%
on the mock data). `decorr_constrained_pca` adds two things:

1. **Oblique-projection scores.** Because the prior patterns are
   non-orthogonal, the correct factor exposure is the OLS regression
   $F = X V (V^\top V)^{-1}$, not the naive dot product $XV$. The
   dot-product version double-counts the overlap between patterns.
2. **Explicit score-decorrelation penalty.** The objective adds
   $\lambda_2 \| \mathrm{offdiag}(\mathrm{Corr}(F)) \|_F^2$
   directly on the correlation matrix (scale-invariant), so the
   patterns are pushed toward distinct market directions while still
   anchored at $V_0$.

Full objective:
$$\min_V \; \|X - F V^\top\|_F^2 + \lambda_1 \|V - V_0\|_F^2 + \lambda_2 \| \mathrm{offdiag}\,\mathrm{Corr}(F) \|_F^2,$$
where $F = X V (V^\top V + \varepsilon I)^{-1}$. Optimised through
PyTorch autodiff (Adam by default).

Comparison with `soft_constrained_pca`:
* Same anchor toward $V_0$.
* New: explicit penalty on score correlation, not just on $V$.
* New: oblique projection — $F$ is determined by $V$ rather than a
  free variable in joint ALS. This is what makes the per-pattern
  exposure interpretable in a multi-collinear basis.

Reference baseline: **orthogonal Procrustes** rotates the top-$k$ PCs
to align with $V_0$. Loadings are exactly orthonormal — so it gives
the *upper bound* on R² for any $V$ that stays inside the top-$k$ PC
subspace. (Orthonormal loadings do NOT give zero score correlation
— after rotation the score covariance is $R^\top \mathrm{diag}(\sigma_i^2) R$.)

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pca import load_long, to_wide, EXPIRY_LABELS, TENOR_LABELS
from factors import (
    soft_constrained_pca,
    decorr_constrained_pca,
    procrustes_pca_baseline,
)

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

vol = to_wide(load_long("../data/mock/atm_vol.pkl")).diff().dropna()
T, P = vol.shape
print(f"vol diff: T={T} days, p={P} cells")

priors_path = Path("../data/priors.pkl")
if not priors_path.exists():
    raise FileNotFoundError(
        f"No prior file at {priors_path.resolve()}. "
        f"Run `streamlit run streamlit_apps/pattern_creator.py` and click Save."
    )
V0 = pd.read_pickle(priors_path).reindex(columns=vol.columns, fill_value=0.0)
k = len(V0)
print(f"loaded {k} priors:", list(V0.index))

def loading_heatmap(ax, vec, title, vmax, cbar=False):
    series = pd.Series(vec, index=vol.columns)
    grid = series.unstack("tenor").reindex(
        index=[e for e in EXPIRY_LABELS if e in vol.columns.get_level_values("expiry")],
        columns=[t for t in TENOR_LABELS if t in vol.columns.get_level_values("tenor")],
    )
    sns.heatmap(grid, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, cbar=cbar)
    ax.set_title(title); ax.set_xlabel("Tenor"); ax.set_ylabel("Expiry")

def score_corr(F: pd.DataFrame) -> pd.DataFrame:
    arr = F.values - F.values.mean(axis=0)
    C = arr.T @ arr / arr.shape[0]
    d = np.sqrt(np.maximum(np.diag(C), 1e-12))
    return pd.DataFrame(C / np.outer(d, d), index=F.columns, columns=F.columns)

def max_offdiag(M: pd.DataFrame) -> float:
    A = M.values.copy(); np.fill_diagonal(A, 0.0)
    return float(np.abs(A).max()) if A.size > 1 else 0.0

## 1. The loading-collapse problem

Fit the existing `soft_constrained_pca` at a moderate `lam`. Look
at the implied score time series. Even though the prior patterns
$V_0$ are visually distinct, the fitted scores end up highly
correlated because joint ALS lets $V$ rotate freely toward the
dominant market direction.

In [ ]:
old_fit = soft_constrained_pca(vol, V0, lam=2.0, init="prior",
                                max_iter=300, tol=1e-7)
F_old = old_fit["F"]
corr_old = score_corr(F_old)
print(f"soft_constrained_pca (lam=2.0):\n  max|Corr(F)| = {max_offdiag(corr_old):.4f}")
print("  full Corr(F):")
print(corr_old.round(3))

## 2. New objective — smoke tests

**Smoke 1.** `lam_decorr = 0` should behave roughly like the old
soft fit (modulo the oblique-projection difference) — i.e. R² in
the same ball-park, max|corr| not driven to zero.

**Smoke 2.** Large `lam_decorr` should crush `max|Corr(F)|` toward
zero, paying a small R² penalty.

In [ ]:
def summarise(fit, label):
    return {
        "label": label,
        "R2": fit["R2"],
        "max|corr|": fit["max_offdiag_corr"],
        "mean drift": float(fit["drift"]["||v - v0||"].mean()),
        "cond(VtV)": fit["cond_VtV"],
        "iters": len(fit["objective_history"]),
    }

smoke1 = decorr_constrained_pca(vol, V0, lam_anchor=2.0, lam_decorr=0.0,
                                 max_iter=400, lr=5e-3, standardize=False)
smoke2 = decorr_constrained_pca(vol, V0, lam_anchor=2.0, lam_decorr=5e4,
                                 max_iter=800, lr=5e-3, standardize=False)
print(pd.DataFrame([summarise(smoke1, "lam_decorr=0"),
                    summarise(smoke2, "lam_decorr=5e4")]).round(4).to_string(index=False))
for fit, lbl in [(smoke1, "lam_decorr=0"), (smoke2, "lam_decorr=5e4")]:
    if fit["warnings"]:
        print(f"\n[{lbl}] warnings:")
        for w in fit["warnings"]:
            print(" ", w)

## 3. Moderate $(\lambda_1, \lambda_2)$ — full diagnostics

Pick a balanced point: enough anchor to keep the patterns visually
recognisable, enough decorr to keep the score correlations small.
Inspect per-pattern drift, the $k \times k$ correlation matrix,
and the conditioning of $V^\top V$.

In [ ]:
fit = decorr_constrained_pca(
    vol, V0,
    lam_anchor=2.0, lam_decorr=5e3,
    max_iter=800, lr=5e-3, standardize=False,
)
print(f"R^2 (reconstruction)     = {fit['R2']:.4f}")
print(f"max|Corr(F)|             = {fit['max_offdiag_corr']:.4f}")
print(f"cond(V^T V)              = {fit['cond_VtV']:.3e}")
print(f"reconstruction_error     = {fit['reconstruction_error']:.2f}")
print()
print("Per-pattern drift:")
print(fit["drift"].round(4).to_string(index=False))
print()
print("Corr(F):")
print(fit["corr_F"].round(3))
if fit["warnings"]:
    print("\nWarnings:")
    for w in fit["warnings"]:
        print(" ", w)

# Loss trajectory.
hist = fit["objective_history"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist["iter"], hist["recon"], label="recon", lw=1.2)
ax.plot(hist["iter"], hist["anchor"] * 2.0, label=r"$\lambda_1 \cdot$ anchor", lw=1.2)
ax.plot(hist["iter"], hist["decorr"] * 5e3, label=r"$\lambda_2 \cdot$ decorr", lw=1.2)
ax.plot(hist["iter"], hist["total"], label="total", color="k", lw=1.5)
ax.set_yscale("log"); ax.set_xlabel("iter"); ax.set_ylabel("loss component")
ax.set_title("Loss components vs iteration"); ax.legend()
plt.tight_layout(); plt.show()

## 4. Loading heatmaps — prior vs old soft vs new decorr

Each pattern is rendered as a `(expiry, tenor)` heatmap and
normalised to unit L2 norm for visual fairness.

In [ ]:
def unit(v):
    n = np.linalg.norm(v); return v / n if n > 0 else v

fig, axes = plt.subplots(k, 3, figsize=(15, 4.5 * k), squeeze=False)
for j, name in enumerate(V0.index):
    v0 = unit(V0.loc[name].values)
    vo = unit(old_fit["V"].loc[name].values)
    vd = unit(fit["V"].loc[name].values)
    vmax = max(np.abs(v0).max(), np.abs(vo).max(), np.abs(vd).max(), 1e-9)
    loading_heatmap(axes[j, 0], v0, f"{name} — prior $V_0$", vmax)
    loading_heatmap(axes[j, 1], vo, f"{name} — soft_constrained", vmax)
    loading_heatmap(axes[j, 2], vd, f"{name} — decorr (this)", vmax)
plt.tight_layout(); plt.show()

## 5. Score time series

Overlay the $k$ score time series for the old and new fits. The
old fit's lines move together (high cross-correlation); the new
fit's lines are independent.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for name in V0.index:
    axes[0].plot(F_old.index, F_old[name].values, label=name, lw=0.9)
    axes[1].plot(fit["F"].index, fit["F"][name].values, label=name, lw=0.9)
axes[0].set_title(f"soft_constrained_pca  |  max|Corr(F)| = {max_offdiag(corr_old):.3f}")
axes[1].set_title(f"decorr_constrained_pca  |  max|Corr(F)| = {fit['max_offdiag_corr']:.3f}")
for ax in axes:
    ax.legend(loc="upper right", fontsize=8); ax.axhline(0, color="k", lw=0.5)
axes[1].set_xlabel("date")
plt.tight_layout(); plt.show()

## 6. Orthogonal-Procrustes baseline

Rotate the top-$k$ PCs onto $V_0$. Loadings are exactly
orthonormal, so this is the **upper bound on R²** for any $V$ that
stays inside the top-$k$ PC subspace. The score correlations are
generally *not* zero — the rotation mixes PC variances.

In [ ]:
base = procrustes_pca_baseline(vol, V0)
print("Procrustes baseline:")
print(f"  R^2          = {base['R2']:.4f}")
print(f"  max|Corr(F)| = {base['max_offdiag_corr']:.4f}")
print("  Corr(F):")
print(base["corr_F"].round(3))

## 7. $(\lambda_1, \lambda_2)$ Pareto sweep

Sweep `lam_anchor` and `lam_decorr` on a grid; record
$(R^2, \max|\mathrm{Corr}(F)|, \text{mean drift})$ for each
combination. The desk picks a point by eye — there's no single
right answer.

* Top-left corner = high R², high correlation (fit hard, tolerate
  collapse).
* Bottom-right = low R², low correlation (decorr-dominated).
* Procrustes is plotted as a reference point.

In [ ]:
lam1_grid = [0.5, 2.0, 10.0]
lam2_grid = [0.0, 5e2, 5e3, 5e4, 5e5]

rows = []
for lam1 in lam1_grid:
    for lam2 in lam2_grid:
        f = decorr_constrained_pca(
            vol, V0, lam_anchor=lam1, lam_decorr=lam2,
            max_iter=400, lr=5e-3, standardize=False,
        )
        rows.append({
            "lam_anchor": lam1, "lam_decorr": lam2,
            "R2": f["R2"],
            "max|corr|": f["max_offdiag_corr"],
            "mean drift": float(f["drift"]["||v - v0||"].mean()),
            "cond(VtV)": f["cond_VtV"],
        })
pareto = pd.DataFrame(rows)
print(pareto.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: R^2 vs max|corr|, coloured by lam_decorr, sized by mean drift.
ax = axes[0]
for lam1 in lam1_grid:
    sub = pareto[pareto["lam_anchor"] == lam1].sort_values("max|corr|")
    ax.plot(sub["max|corr|"], sub["R2"], marker="o", lw=1.2,
            label=f"$\\lambda_1$={lam1}")
    for _, r in sub.iterrows():
        ax.annotate(f"$\\lambda_2$={r['lam_decorr']:g}",
                    (r["max|corr|"], r["R2"]),
                    textcoords="offset points", xytext=(5, 4), fontsize=7)
ax.scatter([base["max_offdiag_corr"]], [base["R2"]],
           marker="*", s=220, color="red", zorder=5, label="Procrustes")
ax.set_xlabel(r"$\max_{i \neq j} \,|\mathrm{Corr}(F)_{ij}|$")
ax.set_ylabel(r"$R^2$")
ax.set_title("Pareto: fit quality vs loading independence")
ax.legend(loc="lower right", fontsize=8)

# Panel B: mean drift vs max|corr|.
ax = axes[1]
for lam1 in lam1_grid:
    sub = pareto[pareto["lam_anchor"] == lam1].sort_values("max|corr|")
    ax.plot(sub["max|corr|"], sub["mean drift"], marker="o", lw=1.2,
            label=f"$\\lambda_1$={lam1}")
ax.set_xlabel(r"$\max_{i \neq j} \,|\mathrm{Corr}(F)_{ij}|$")
ax.set_ylabel(r"mean $\|v_j - v_j^{(0)}\|$")
ax.set_title("Decorrelating costs pattern drift")
ax.legend(loc="upper right", fontsize=8)

plt.tight_layout(); plt.show()

## Notes

* The decorrelation penalty is on the **correlation** matrix
  (scale-invariant), not on the covariance. Penalising covariance
  off-diagonals would also collapse to the same correlation
  objective only if loadings are unit-normed — which they aren't here.
* The oblique projection $F = X V (V^\top V)^{-1}$ is what makes
  per-pattern exposures interpretable when patterns overlap. If
  `cond(V^T V)` is large the function prints a warning; bump `eps`
  or `lam_decorr` in that case.
* `lam_anchor` and `lam_decorr` are raw weights on the penalty
  terms — not N-scaled. Reconstruction is on the order of
  $T \cdot \mathrm{tr}(\Sigma)$, anchor on $O(k)$, and decorr on
  $O(k^2)$, so for moderate priors you typically need
  $\lambda_2 \sim 10^3$–$10^5$ to compete with reconstruction. Start
  from the Pareto sweep above and zoom in.
* The Procrustes reference is a single point, not a frontier. It
  shows the best R² achievable when $V$ is constrained to the
  top-$k$ PC subspace and orthonormal — useful as an upper bound
  but it generally does NOT minimise max|corr|.